# 🏙️ Spanish Cities Real Estate Market Analysis

**Purpose:** Comprehensive Spanish real estate market analysis across multiple cities  
**Dataset:** Spanish Real Estate Markets (All Available Cities)  
**Date:** December 2024  
**Environment:** Python 3.8+, pandas, numpy, plotly, matplotlib, seaborn

## 🎯 Analysis Objectives

- Automatically discover and process aall available cities with housing data
- Analyze sales and rental markets for each city independently
- Generate comprehensive market profiles for each urban area
- Perform cross-city comparative analysis and rankings
- Identify national market patterns and investment opportunities
- Export consolidated datasets ready for predictive modeling

## 📋 Metadata

- **Purpose:** Multi-city exploratory and comparative real estate analysis
- **Dataset version:** Raw Kaggle data - All available Spanish cities
- **Required environment:** Python 3.8+, pandas>=1.3.0, numpy>=1.21.0, matplotlib>=3.5.0, seaborn>=0.11.0
- **Date:** December 2024
- **Processing scope:** Automatically discovered from houses_*.csv files
- **Hardware requirements:** Minimum 4GB RAM for processing all cities


In [148]:
"""
Environment Setup and Configuration
"""
import sys
import warnings
from pathlib import Path

# Add project root to path for module imports
project_root = Path('..').resolve()
sys.path.insert(0, str(project_root))

# Data science libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Project modules
from src.data.exploratory_analysis import (
    load_data, clean_data, split_data, save_data, 
    create_mapping_dataframe, generate_basic_profile, 
    harmonize_datasets
)
from src.analysis.comparative_analysis import (
    safe_compare_markets, examine_data_structure
)

# Configuration
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', lambda x: '%.2f' % x)
sns.set_style("whitegrid")

# Try to import plotly with fallback
try:
    import plotly.express as px
    import plotly.graph_objects as go
    HAS_PLOTLY = True
    px.defaults.template = "plotly_white" 
    px.defaults.width = 1200
    px.defaults.height = 700
    print("✅ Plotly available for enhanced visualizations")
except ImportError:
    HAS_PLOTLY = False
    print("⚠️ Plotly not available. Using matplotlib for visualizations.")

# Define paths
PROVINCE = 'zamora'
DATA_PATH = Path('../data/raw/data-kaggle')
PROCESSED_PATH = Path('../data/processed/'+ PROVINCE)
FINAL_PATH = Path('../data/final/'+ PROVINCE)
REPORTS_PATH = Path('../reports/profiles')

# Create directories
for path in [PROCESSED_PATH, FINAL_PATH, REPORTS_PATH]:
    path.mkdir(parents=True, exist_ok=True)

print("\n✅ Environment setup complete!")
print(f"📁 Raw data path: {DATA_PATH}")
print(f"📁 Processed data path: {PROCESSED_PATH}")
print(f"📁 Final data path: {FINAL_PATH}")
print(f"💾 Reports path: {REPORTS_PATH}")


✅ Plotly available for enhanced visualizations

✅ Environment setup complete!
📁 Raw data path: ../data/raw/data-kaggle
📁 Processed data path: ../data/processed/zamora
📁 Final data path: ../data/final/zamora
💾 Reports path: ../reports/profiles


## 1. Data Loading and Initial Processing

First, we'll load and process the Álava housing data using our standardized pipeline.


In [149]:
# Load and process Álava housing data
province_file = DATA_PATH / f"houses_{PROVINCE}.csv"
print(f"🔄 Processing data from: {province_file}")

if not province_file.exists():
    print(f"❌ File not found: {province_file}")
    sys.exit(1)

# Load the raw data
raw_data = load_data(province_file)

if raw_data is None:
    print("❌ Failed to load data. Exiting...")
    sys.exit(1)

# Clean the data
cleaned_data, house_type_mapping = clean_data(raw_data)

if cleaned_data is None:
    print("❌ Failed to clean data. Exiting...")
    sys.exit(1)

# Split into rental and sales data
rental_data, sales_data = split_data(cleaned_data, house_type_mapping)

# Save processed datasets
if rental_data is not None:
    save_data(rental_data, PROCESSED_PATH / f"houses_{PROVINCE}_cleaned_rent.csv")

if sales_data is not None:
    save_data(sales_data, PROCESSED_PATH / f"houses_{PROVINCE}_cleaned_sale.csv")

# Save house type mapping
if house_type_mapping:
    mapping_df = create_mapping_dataframe(house_type_mapping, 'House_Type_Mapping')
    save_data(mapping_df, PROCESSED_PATH / 'houses_type_mapping.csv')

# Set variables for analysis
df_sales = sales_data
df_rental = rental_data

print(f"\n✅ Data processing complete!")
print(f"   Sales data: {df_sales.shape if df_sales is not None else 'None'}")
print(f"   Rental data: {df_rental.shape if df_rental is not None else 'None'}")


🔄 Processing data from: ../data/raw/data-kaggle/houses_zamora.csv
✅ Data loaded successfully from ../data/raw/data-kaggle/houses_zamora.csv
   Shape: (3394, 36)
   Memory usage: 7.65 MB
🧹 Cleaning and preprocessing data...
   Dropped columns: ['ground_size', 'kitchen', 'unfurnished', 'loc_street', 'ad_description']
   Applied one-hot encoding to: ['condition', 'heating', 'orientation']
✅ Data cleaned successfully. Final shape: (3394, 57)
📊 Data split successfully:
   Rental data: 163 records
   Sales data: 3231 records
💾 Data saved successfully to ../data/processed/zamora/houses_zamora_cleaned_rent.csv
   Shape: (163, 57)
💾 Data saved successfully to ../data/processed/zamora/houses_zamora_cleaned_sale.csv
   Shape: (3231, 57)
💾 Data saved successfully to ../data/processed/zamora/houses_type_mapping.csv
   Shape: (24, 2)

✅ Data processing complete!
   Sales data: (3231, 57)
   Rental data: (163, 57)


## 2. Data Structure Examination

Let's examine the structure and content of our processed datasets.


In [150]:
# Examine data structures
if df_sales is not None:
    examine_data_structure(df_sales, "Sales Data")


=== SALES DATA STRUCTURE ===
Shape: (3231, 57)
Memory usage: 4.47 MB

Columns (57):
  - ad_last_update: object (3231 non-null, 0 null)
  - air_conditioner: object (3231 non-null, 0 null)
  - balcony: object (3231 non-null, 0 null)
  - bath_num: float64 (3231 non-null, 0 null)
  - built_in_wardrobe: object (3231 non-null, 0 null)
  - chimney: object (3231 non-null, 0 null)
  - construct_date: object (1209 non-null, 2022 null)
  - energetic_certif: object (2539 non-null, 692 null)
  - floor: object (2664 non-null, 567 null)
  - garage: int64 (3231 non-null, 0 null)
  - garden: object (3231 non-null, 0 null)
  - house_id: object (3231 non-null, 0 null)
  - house_type: int64 (3231 non-null, 0 null)
  - lift: object (1849 non-null, 1382 null)
  - loc_city: object (3231 non-null, 0 null)
  - loc_district: object (2973 non-null, 258 null)
  - loc_full: object (3231 non-null, 0 null)
  - loc_neigh: object (888 non-null, 2343 null)
  - loc_zone: object (3231 non-null, 0 null)
  - m2_real: obje

In [151]:
if df_rental is not None:
    examine_data_structure(df_rental, "Rental Data")


=== RENTAL DATA STRUCTURE ===
Shape: (163, 57)
Memory usage: 0.23 MB

Columns (57):
  - ad_last_update: object (163 non-null, 0 null)
  - air_conditioner: object (163 non-null, 0 null)
  - balcony: object (163 non-null, 0 null)
  - bath_num: float64 (163 non-null, 0 null)
  - built_in_wardrobe: object (163 non-null, 0 null)
  - chimney: object (163 non-null, 0 null)
  - construct_date: object (31 non-null, 132 null)
  - energetic_certif: object (110 non-null, 53 null)
  - floor: object (148 non-null, 15 null)
  - garage: int64 (163 non-null, 0 null)
  - garden: object (163 non-null, 0 null)
  - house_id: object (163 non-null, 0 null)
  - house_type: int64 (163 non-null, 0 null)
  - lift: object (125 non-null, 38 null)
  - loc_city: object (163 non-null, 0 null)
  - loc_district: object (157 non-null, 6 null)
  - loc_full: object (163 non-null, 0 null)
  - loc_neigh: object (66 non-null, 97 null)
  - loc_zone: object (163 non-null, 0 null)
  - m2_real: object (163 non-null, 0 null)
  -

## 3. Profile Generation

Generate comprehensive profiles for both datasets to understand their characteristics.


In [152]:

# Generate profiles for both datasets
if df_sales is not None:
    sales_profile = generate_basic_profile(df_sales, f"{PROVINCE} Real Estate Sales")
    print("\n" + "="*60)

if df_rental is not None:
    rental_profile = generate_basic_profile(df_rental, f"{PROVINCE} Real Estate Rentals")


📈 Generating basic profile for: zamora Real Estate Sales
   DataFrame shape: (3231, 57)

   === PROFILE SUMMARY ===
   Rows: 3,231
   Columns: 57
   Duplicate rows: 10

   Missing values:
     - construct_date: 2022 (62.58%)
     - energetic_certif: 692 (21.42%)
     - floor: 567 (17.55%)
     - lift: 1382 (42.77%)
     - loc_district: 258 (7.99%)
     - loc_neigh: 2343 (72.52%)
     - m2_useful: 1063 (32.9%)

📈 Generating basic profile for: zamora Real Estate Rentals
   DataFrame shape: (163, 57)

   === PROFILE SUMMARY ===
   Rows: 163
   Columns: 57
   Duplicate rows: 0

   Missing values:
     - construct_date: 132 (80.98%)
     - energetic_certif: 53 (32.52%)
     - floor: 15 (9.2%)
     - lift: 38 (23.31%)
     - loc_district: 6 (3.68%)
     - loc_neigh: 97 (59.51%)
     - m2_useful: 52 (31.9%)


## 4. Data Harmonization and Export

Prepare harmonized datasets for comparative analysis and export to final directory.


In [153]:
# Harmonize and export datasets
if df_sales is not None or df_rental is not None:
    df_sales_final, df_rental_final = harmonize_datasets(
        df_sales, df_rental, FINAL_PATH, PROVINCE
    )
    
    print(f"\n📋 Sample of harmonized sales data:")
    print(df_sales_final.head(3).to_string())
    
    print(f"\n📋 Sample of harmonized rental data:")
    print(df_rental_final.head(3).to_string())
else:
    print("❌ Cannot harmonize datasets: Missing sales or rental data")


✅ Harmonized datasets saved to: ../data/final/zamora
   Sales: 3231 rows × 11 cols
   Rental: 163 rows × 11 cols

📋 Sample of harmonized sales data:
    price  bath_num  room_num  house_type  house_id m2_real m2_useful loc_city               loc_zone construct_date  garage
0   63000      1.00      3.00           0  85017409     109        86     Toro  Alfoz de Toro, Zamora            NaN       0
1  220000      3.00      4.00           1  81263788     360       NaN     Toro  Alfoz de Toro, Zamora            NaN       1
2   60000      1.00      5.00           2  81821822      40       100     Toro  Alfoz de Toro, Zamora            NaN       0

📋 Sample of harmonized rental data:
     price  bath_num  room_num  house_type  house_id m2_real m2_useful                     loc_city                loc_zone construct_date  garage
1347   320      1.00      1.00          15  81670307      60       NaN                       Zamora  Área de Zamora, Zamora            NaN       1
1348   360      2.00

## 5. Comparative Market Analysis

Now let's perform a comprehensive comparison between the sales and rental markets.


In [154]:
# Perform comprehensive market comparison
if df_sales is not None and df_rental is not None:
    print("🔍 Starting comprehensive market comparison...")
    
    # Use the safe comparison function that handles all edge cases
    comparison_success = safe_compare_markets(
        df_sales, 
        df_rental,
        price_col='price',  # Let the function auto-detect if None
        surface_col=None    # Let the function auto-detect
    )
    
    if comparison_success:
        print("\n✅ Market comparison completed successfully!")
    else:
        print("\n⚠️ Market comparison had issues. Check data structure.")
        # Fallback: basic comparison
        print("\n=== BASIC COMPARISON FALLBACK ===")
        print(f"Sales data: {df_sales.shape[0]} records with {df_sales.shape[1]} features")
        print(f"Rental data: {df_rental.shape[0]} records with {df_rental.shape[1]} features")
else:
    print("❌ Cannot compare markets: Missing data for either sales or rentals")


🔍 Starting comprehensive market comparison...

=== EXAMINING COLUMNS FOR COMPARISON ===
Price column: price
Surface columns available: ['m2_real', 'm2_useful']
Selected surface column: m2_real

Sales data has required columns: True
Rental data has required columns: True
❌ Error in market comparison: unsupported operand type(s) for /: 'str' and 'str'

⚠️ Market comparison had issues. Check data structure.

=== BASIC COMPARISON FALLBACK ===
Sales data: 3231 records with 57 features
Rental data: 163 records with 57 features


## 6. Summary and Conclusions

Key insights from the Álava real estate market analysis.


In [155]:
# Summary of analysis
print(f"📊 {PROVINCE.upper()} REAL ESTATE MARKET ANALYSIS SUMMARY")
print("="*50)

if df_sales is not None and df_rental is not None:
    print(f"✅ Analysis completed successfully")
    print(f"   📈 Sales market: {df_sales.shape[0]:,} properties analyzed")
    print(f"   🏠 Rental market: {df_rental.shape[0]:,} properties analyzed")
    print(f"   📁 Harmonized datasets exported to: {FINAL_PATH}")
    print(f"   🔍 Processed data saved to: {PROCESSED_PATH}")
    
    # Key metrics summary
    if 'price' in df_sales.columns and 'price' in df_rental.columns:
        avg_sale_price = df_sales['price'].mean()
        avg_rental_price = df_rental['price'].mean()
        print(f"\n💰 Key Price Metrics:")
        print(f"   Average sales price: €{avg_sale_price:,.2f}")
        print(f"   Average monthly rental: €{avg_rental_price:,.2f}")
        print(f"   Annual rental equivalent: €{avg_rental_price * 12:,.2f}")
        
        if avg_sale_price > 0:
            estimated_yield = (avg_rental_price * 12) / avg_sale_price * 100
            print(f"   Estimated gross rental yield: {estimated_yield:.2f}%")
    else:
        print("❌ Analysis incomplete due to missing price columns")
else:
    print("❌ Analysis incomplete due to data processing issues")

print(f"\n📋 Files generated:")
print(f"   - {FINAL_PATH}/{PROVINCE}_sales_final.csv")
print(f"   - {FINAL_PATH}/{PROVINCE}_rental_final.csv") 
print(f"   - {FINAL_PATH}/{PROVINCE}_sales_describe.csv")
print(f"   - {FINAL_PATH}/{PROVINCE}_rental_describe.csv")
print(f"   - {PROCESSED_PATH}/houses_type_mapping.csv")

print(f"\n🎯 Analysis objectives achieved:")
print(f"   ✅ Datasets processed and cleaned")
print(f"   ✅ Market profiles generated") 
print(f"   ✅ Comparative analysis performed")
print(f"   ✅ Data harmonized and exported")
print(f"   ✅ Ready for predictive modeling")


📊 ZAMORA REAL ESTATE MARKET ANALYSIS SUMMARY
✅ Analysis completed successfully
   📈 Sales market: 3,231 properties analyzed
   🏠 Rental market: 163 properties analyzed
   📁 Harmonized datasets exported to: ../data/final/zamora
   🔍 Processed data saved to: ../data/processed/zamora


TypeError: Could not convert string '6300022000060000580003500090000600055000680004000089000120000300002990006000045000910001700002500065000280009000060000120000624001432509000050000632009600022000560000250001500001100002475015000168000700001000002405010850035000850003400012000800005800065000100000120000420002000002000080004700085000255000385000450008500035000070000650005800040000550001990049000150002500038000125000400001100078000105000170000420000410007950012000012900012200011000600006000025500449614000089000285004400020000048900530001000008300083000830008300010000083000830008300083000830001000001980004400048000240008000012000048900168000210001500001950005000048000125000180000600004000038900590004800010000024000490001190001990075000180000115000300006800055000750003630010850033300060000198000630007500040000449617500045000128500600004950180000850002100007090068500758007145018000650000150000320005500036300389004215071450758006850070900price387002600030000300001500048000115000600001000004